In [1]:
import json
import pandas as pd
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPEN_API_KEY")  
client = OpenAI(api_key=api_key)



In [2]:
import sqlite3

def create_database():
    conn = sqlite3.connect("company.db")  # creates company.db file
    cursor = conn.cursor()

    # SQLite uses INTEGER PRIMARY KEY instead of AUTO_INCREMENT
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS employees (
            employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name VARCHAR(50) NOT NULL,
            last_name VARCHAR(50) NOT NULL,
            department_id INT,
            salary DECIMAL(10, 2),
            hire_date DATE,
            manager_id INT,
            email VARCHAR(100),
            phone_number VARCHAR(20),
            job_title VARCHAR(50)
        )
    """)

    cursor.executemany("""
        INSERT OR IGNORE INTO employees 
        (first_name, last_name, department_id, salary, hire_date, manager_id, email, phone_number, job_title)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        ('John', 'Doe', 1, 60000, '2018-01-15', None, 'john.doe@example.com', '555-1234', 'Manager'),
        ('Jane', 'Smith', 2, 50000, '2019-03-22', 1, 'jane.smith@example.com', '555-5678', 'Developer'),
        ('Michael', 'Johnson', 1, 45000, '2020-06-11', 1, 'michael.johnson@example.com', '555-8765', 'Analyst'),
        ('Emily', 'Davis', 3, 55000, '2017-09-30', None, 'emily.davis@example.com', '555-3456', 'Sales'),
        ('Daniel', 'Wilson', 3, 40000, '2018-11-10', 4, 'daniel.wilson@example.com', '555-6543', 'Sales Representative'),
        ('Sophia', 'Martinez', 2, 47000, '2021-01-25', 2, 'sophia.martinez@example.com', '555-7890', 'Developer'),
        ('James', 'Brown', 1, 38000, '2020-04-18', 1, 'james.brown@example.com', '555-1239', 'Support'),
        ('Olivia', 'Jones', 2, 62000, '2019-07-21', 1, 'olivia.jones@example.com', '555-4321', 'Lead Developer'),
        ('William', 'Garcia', 3, 52000, '2018-12-01', None, 'william.garcia@example.com', '555-5670', 'Sales Manager'),
        ('Isabella', 'Miller', 2, 48000, '2020-08-15', 2, 'isabella.miller@example.com', '555-6789', 'QA Engineer'),
        ('Alex', 'White', 1, 61000, '2020-11-23', 1, 'alex.white@example.com', '555-2345', 'HR Specialist'),
        ('Liam', 'Lee', 2, 55000, '2017-05-19', 8, 'liam.lee@example.com', '555-3457', 'Developer'),
        ('Emma', 'Clark', 3, 58000, '2021-03-14', 4, 'emma.clark@example.com', '555-4568', 'Sales Executive'),
        ('Noah', 'Lopez', 1, 43000, '2019-12-10', 1, 'noah.lopez@example.com', '555-5679', 'HR Assistant'),
        ('Ava', 'Gonzalez', 2, 49000, '2018-07-25', 2, 'ava.gonzalez@example.com', '555-6780', 'QA Analyst'),
        ('Mason', 'Harris', 3, 61000, '2016-08-29', 4, 'mason.harris@example.com', '555-7891', 'Senior Sales Manager'),
        ('Ethan', 'Walker', 1, 42000, '2021-02-10', 1, 'ethan.walker@example.com', '555-8901', 'HR Coordinator'),
        ('Mia', 'Young', 2, 57000, '2018-09-15', 8, 'mia.young@example.com', '555-9012', 'Developer'),
        ('Logan', 'Hall', 3, 45000, '2020-05-17', 4, 'logan.hall@example.com', '555-1230', 'Sales Associate'),
        ('Charlotte', 'Allen', 2, 53000, '2019-01-20', 8, 'charlotte.allen@example.com', '555-2341', 'UI/UX Designer'),
        ('Benjamin', 'King', 1, 47000, '2017-11-22', 1, 'benjamin.king@example.com', '555-3452', 'HR Manager'),
        ('Amelia', 'Wright', 2, 50000, '2020-06-13', 8, 'amelia.wright@example.com', '555-4563', 'Software Tester'),
        ('Lucas', 'Scott', 3, 49000, '2016-10-30', 4, 'lucas.scott@example.com', '555-5674', 'Sales Coordinator'),
        ('Ella', 'Green', 1, 53000, '2018-02-14', 1, 'ella.green@example.com', '555-6785', 'HR Specialist'),
        ('Aiden', 'Adams', 2, 51000, '2019-08-26', 8, 'aiden.adams@example.com', '555-7896', 'Backend Developer'),
        ('Grace', 'Baker', 3, 52000, '2021-04-21', 4, 'grace.baker@example.com', '555-8907', 'Sales Analyst'),
        ('Oliver', 'Nelson', 1, 45000, '2020-01-19', 1, 'oliver.nelson@example.com', '555-9018', 'HR Associate'),
        ('Avery', 'Carter', 2, 60000, '2017-12-11', 8, 'avery.carter@example.com', '555-1239', 'Front-End Developer'),
        ('Matthew', 'Mitchell', 3, 48000, '2018-06-07', 4, 'matthew.mitchell@example.com', '555-2340', 'Sales Representative'),
        ('Harper', 'Perez', 2, 56000, '2019-03-28', 8, 'harper.perez@example.com', '555-3451', 'Quality Assurance'),
        ('Alice', 'Wonder', 2, 45000, '2022-07-13', 2, 'alice.wonderexample.com', '555-9999', 'Intern'),
        ('Bob', 'Builder', 3, 48000, '2021-11-10', 3, 'bob%builder@example.com', '555-8888', 'Technician'),
    ])

    conn.commit()
    conn.close()
    print("✅ Database created and populated.")

create_database()

✅ Database created and populated.


In [3]:
def get_connection():
    return sqlite3.connect("company.db")

In [21]:
def get_schema() -> str:
    conn = get_connection()
    cursor = conn.cursor()

    # get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]

    schema = ""
    for table in tables:
        cursor.execute(f"PRAGMA table_info({table})")
        columns = cursor.fetchall()
        schema += f"\nTable: {table}\nColumns:\n"
        for col in columns:
            # col = (cid, name, type, notnull, default, pk)
            schema += f"  - {col[1]} ({col[2]})\n"

    conn.close()
    return schema

# verify
print(get_schema())


Table: employees
Columns:
  - employee_id (INTEGER)
  - first_name (VARCHAR(50))
  - last_name (VARCHAR(50))
  - department_id (INT)
  - salary (DECIMAL(10, 2))
  - hire_date (DATE)
  - manager_id (INT)
  - email (VARCHAR(100))
  - phone_number (VARCHAR(20))
  - job_title (VARCHAR(50))

Table: sqlite_sequence
Columns:
  - name ()
  - seq ()



In [22]:
import re

In [23]:
def generate_sql(question:str,schema:str,model:str) -> str:
     prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
     response = client.chat.completions.create(
        model=model,
        max_tokens= 1024,
        messages=[{"role":"user","content":prompt}]
    )

     sql = response.choices[0].message.content.strip()
     sql = re.sub(r"```sql", "", sql)
     sql = re.sub(r"```", "", sql)
     return sql.strip()
    


In [ ]:
Schema = """
Table: employees
Columns:
  - employee_id (INTEGER)
  - first_name (VARCHAR(50))
  - last_name (VARCHAR(50))
  - department_id (INT)
  - salary (DECIMAL(10, 2))
  - hire_date (DATE)
  - manager_id (INT)
  - email (VARCHAR(100))
  - phone_number (VARCHAR(20))
  - job_title (VARCHAR(50))
  """

question = "employee with second highest salary"
model = "gpt-4o-mini"


In [25]:
def run_sql(sql_query: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(sql_query)
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    conn.close()
    return [dict(zip(columns, row)) for row in rows]

In [27]:
sql_query = generate_sql(question,Schema,model)
print(sql_query)

SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);


In [28]:
results = run_sql(sql_query)
print("Results:", results)

Results: [{'second_highest_salary': 61000}]
